# Publish the fine-tuned model to the Hugging Face Hub

This takes the weights we trained (saved on Drive at
`weights/codereviewer-msg-pyjs/`) and pushes them to a **public** Hugging Face
model repo. Once published, the review pipeline (and the live demo) can load the
model by id — e.g. `AI_MODEL_PATH=<your-username>/codereviewer-msg-pyjs` — with
no token needed by the reader.

**One-time human setup (not automatable):**
1. Create a free account at https://huggingface.co
2. Create a **write** access token: Settings → Access Tokens → *New token* (role: Write)
3. Run this notebook on Colab and paste the token when prompted.

Run top-to-bottom on any runtime (CPU is fine — this only uploads files).


## 0. Install


In [1]:
!pip -q install "transformers==4.44.2" "huggingface_hub>=0.24"
import transformers, huggingface_hub
print("transformers:", transformers.__version__, "| hub:", huggingface_hub.__version__)


transformers: 4.44.2 | hub: 0.36.2


## 1. Mount Drive and locate the trained weights


In [2]:
from pathlib import Path
from google.colab import drive
drive.mount("/content/drive")

ROOT = Path("/content/drive/MyDrive/code-review-ai")
WEIGHTS_DIR = ROOT / "weights" / "codereviewer-msg-pyjs"

assert WEIGHTS_DIR.exists(), (
    f"Weights not found at {WEIGHTS_DIR}. If you deleted them, re-run the eval "
    f"notebook (05) to re-save from a checkpoint, or point WEIGHTS_DIR at the "
    f"right folder."
)
have = sorted(p.name for p in WEIGHTS_DIR.iterdir())
print("Found weights folder with:", have)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Found weights folder with: ['added_tokens.json', 'config.json', 'generation_config.json', 'merges.txt', 'model.safetensors', 'special_tokens_map.json', 'tokenizer.json', 'tokenizer_config.json', 'training_args.bin', 'vocab.json']


## 2. Choose the target model id


In [3]:
# EDIT THIS: use your Hugging Face username.
HF_USERNAME = "Jabez0701"        # <-- change me
MODEL_NAME  = "codereviewer-msg-pyjs"
HF_MODEL_ID = f"{HF_USERNAME}/{MODEL_NAME}"
print("Will publish to:", f"https://huggingface.co/{HF_MODEL_ID}")
assert HF_USERNAME != "your-username", "Set HF_USERNAME to your real HF username first."


Will publish to: https://huggingface.co/Jabez0701/codereviewer-msg-pyjs


## 3. Log in with a WRITE token


In [4]:
from huggingface_hub import login
from getpass import getpass
# Token is read privately and NOT stored in the notebook.
login(getpass("Paste your Hugging Face WRITE token: "))
print("Logged in.")


Paste your Hugging Face WRITE token: ··········
Logged in.


## 4. Load the fine-tuned model + tokenizer, then push


In [6]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(str(WEIGHTS_DIR))
model = AutoModelForSeq2SeqLM.from_pretrained(str(WEIGHTS_DIR))
print("Loaded model with", sum(p.numel() for p in model.parameters())//1_000_000, "M params")

# Push both. private=False makes it public (readers need no token).
model.push_to_hub(HF_MODEL_ID, private=False)
tokenizer.push_to_hub(HF_MODEL_ID, private=False)
print("Pushed ->", f"https://huggingface.co/{HF_MODEL_ID}")


Loaded model with 222 M params


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...qqrisyd/model.safetensors:   0%|          |  554kB /  892MB            

README.md: 0.00B [00:00, ?B/s]

Pushed -> https://huggingface.co/Jabez0701/codereviewer-msg-pyjs


## 5. Add a short model card (README on the Hub)


In [7]:
from huggingface_hub import ModelCard

card_text = f'''---
license: apache-2.0
language:
- code
tags:
- code-review
- codet5
- code-language-model
pipeline_tag: text2text-generation
---

# {MODEL_NAME}

`microsoft/codereviewer` fine-tuned on the CodeReviewer **comment-generation**
task (Python + JavaScript). Given a code change (a diff encoded with
`<add>/<del>/<keep>` line markers), it generates a natural-language review comment.

- Base model: `microsoft/codereviewer` (CodeT5 family, pre-trained on diffs+reviews)
- Task: review-comment generation (the CodeReviewer `msg` task)
- Eval: official CodeReviewer smoothed BLEU-4 (`rmstop=True`) = **7.26** on the
  Python/JS test split (reference paper ~5.73).

## Input format
Prefix each diff line with an edit marker, e.g.:
```
<keep> def get_user(conn, username):
<del>     cur.execute("SELECT * FROM users WHERE name = '" + username + "'")
<add>     cur.execute("SELECT * FROM users WHERE name = ?", (username,))
```
'''

ModelCard(card_text).push_to_hub(HF_MODEL_ID)
print("Model card published.")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/hf_api.py:9662: UserWarning: Warnings while validating metadata in README.md:
- The pipeline tag "text2text-generation" is not in the official list: text-classification, token-classification, table-question-answering, question-answering, zero-shot-classification, translation, summarization, feature-extraction, text-generation, fill-mask, sentence-similarity, text-to-speech, text-to-audio, automatic-speech-recognition, audio-to-audio, audio-classification, audio-text-to-text, voice-activity-detection, depth-estimation, image-classification, object-detection, image-segmentation, text-to-image, image-to-text, image-to-image, image-to-video, unconditional-image-generation, video-classification, reinforcement-learning, robotics, tabular-classification, tabular-regression, tabular-to-text, table-to-text, multiple-choice, text-ranking, text-retrieval, time-series-forecasting, text-to-video, image-text-to-text, image-text-to-image, image-

Model card published.


## 6. Verify — load from the Hub and generate one comment


In [8]:
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

tok = AutoTokenizer.from_pretrained(HF_MODEL_ID)
mdl = AutoModelForSeq2SeqLM.from_pretrained(HF_MODEL_ID).eval()

sample = ("<keep> def get_user(conn, username):\n"
          "<del>     cur.execute(\"SELECT * FROM users WHERE name = '\" + username + \"'\")\n"
          "<add>     cur.execute(\"SELECT * FROM users WHERE name = ?\", (username,))")
enc = tok(sample, return_tensors="pt", truncation=True, max_length=512)
with torch.no_grad():
    out = mdl.generate(**enc, max_new_tokens=64, num_beams=4)
print("Review comment:", tok.decode(out[0], skip_special_tokens=True))
print("\nDone. Set AI_MODEL_PATH =", HF_MODEL_ID, "in the GitHub Actions repo variables to enable the AI step.")


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/892M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/163 [00:00<?, ?B/s]

Review comment: I don't think you need the outer parentheses here.

Done. Set AI_MODEL_PATH = Jabez0701/codereviewer-msg-pyjs in the GitHub Actions repo variables to enable the AI step.
